# 01 — Data Acquisition

Download FFIC cases and filter Polymarket trades.

In [ ]:
import subprocess, sys

import pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve()))

In [ ]:
# Step 1: Download FFIC
result = subprocess.run([sys.executable, "src/data/fetch_ffic.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

In [ ]:
# Step 2: Load and inspect FFIC cases
import json
cases = [json.loads(l) for l in open("data/raw/ffic/ffic-v1.jsonl") if l.strip()]
print(f"{len(cases)} cases, {sum(len(c['markets']) for c in cases)} markets")
for c in cases:
    print(f"  {c['case_id']} | {c['date']} | {c['title'][:60]}")

In [ ]:
# Step 3: Filter Polymarket trades (WARNING: will take a long time on full dataset)
# Set DRY_RUN=True to skip this step and use existing cached data
DRY_RUN = False
if not DRY_RUN:
    result = subprocess.run([sys.executable, "src/data/fetch_polymarket.py"], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)
else:
    print("Dry run — skipping download")

In [ ]:
# Step 4: Build market index
result = subprocess.run([sys.executable, "src/data/build_market_index.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

In [ ]:
# Verify
import pandas as pd
idx = pd.read_parquet("data/processed/market_index.parquet")
print(idx.head())
print(f"
Leaked: {idx['is_leaked'].sum()}  Control: {(idx['is_leaked']==0).sum()}")